In [1]:
import sys
sys.path.append(r'X:/')

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import refinitiv.data as rd
import datetime as dt

from Loaders.EikonSpot_class import EikonSpot
from Database.DB_writer import db_writer
from Database.DB_reader import Database

In [3]:
rd.open_session()

<refinitiv.data.session.Definition object at 0x21436089150 {name='workspace'}>

In [4]:
start_date = dt.datetime(2024,11,27)
end_date = dt.datetime(2024,11,28)

spot_inst = EikonSpot('de', start_date, end_date)

spot = spot_inst.spot_data()

df = spot[((spot.index>=dt.datetime(2024,6,5))&
          (spot.index<dt.datetime(2024,6,6)))].copy()
df = df.reset_index()

In [5]:
df['check'] = (df['datetime'] - df['datetime'].shift(1)).dt.seconds/3600

In [6]:
df.loc[df['check']!=1.]

,datetime,b_volume,s_volume,volume,price,check
0,2024-06-05,NaN,NaN,NaN,15.01,NaN


In [ ]:
#['at', 'be', 'cz', 'de', 'dkw', 'dke', 'fr','hu','nl','sk', 'si', 'ro', 'bg']

In [12]:
db = Database()
query = """
SELECT MAX("datetime") 
FROM "spot"."es";
"""
last_date = pd.read_sql(query, db.connection_string).iloc[0].iloc[0]
if not last_date:
    print('No last date')

No last date


In [18]:
dt.normalize(dt.datetime.today())

AttributeError: module 'datetime' has no attribute 'normalize'

In [5]:
start_date = dt.datetime(2019,1,1)
end_date = dt.datetime(2025,1,25)
spot_df = pd.DataFrame()
check_df = pd.DataFrame()
db_r = Database()
# 'at', 'be','de', 'dkw', 'dke', 'fr','nl'
# 'sk','at', 'be','de', 'dkw', 'dke', 'fr','nl', 'cz', 'si',
for country in [ 'es']:
    
    aux_inst = EikonSpot(country, start_date, end_date)
    aux = aux_inst.spot_data()
    df = aux[((aux.index>=dt.datetime(2024,12,13))&
              (aux.index<dt.datetime(2024,12,17)))].copy()
    df = df.reset_index()
    df['check'] = (df['datetime'] - df['datetime'].shift(1)).dt.seconds/3600
    aux_check = df.loc[df['check']!=1.].copy()
    if len(aux_check)>1:
        print('FAILED_CHECK: ', country)
        break
        
    else:
        df.to_sql(name="stage_" + country,
                              schema='spot',
                              con=db_r.connection_string,
                              if_exists='replace', index=False)
        db_r.merge_from_staging_to_prod_enum(schema='spot', table=country)
    

Connected to the database postgre
Table 'es' created in schema 'spot'.


ValueError: 'es' is not in list

In [6]:
aux


,b_volume,s_volume,volume,price
datetime,,,,
2019-01-01 00:00:00,NaN,NaN,NaN,66.88
2019-01-01 01:00:00,NaN,NaN,NaN,66.88
2019-01-01 02:00:00,NaN,NaN,NaN,66.00
2019-01-01 03:00:00,NaN,NaN,NaN,63.64
2019-01-01 04:00:00,NaN,NaN,NaN,58.85
...,...,...,...,...
2025-01-25 19:00:00,NaN,NaN,NaN,91.00
2025-01-25 20:00:00,NaN,NaN,NaN,96.37
2025-01-25 21:00:00,NaN,NaN,NaN,87.38


In [6]:
rd.open_session()
temp = rd.get_data(['0#COAL-FST-EEX'])

C:\Users\krajcovic\AppData\Roaming\Python\Python311\site-packages\refinitiv\data\_tools\_dataframe.py:171:FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`


In [7]:
temp

,Instrument,PROD_PERM,RDNDISPLAY,DSPLY_NAME,RDN_EXCHID,TIMACT,CURRENCY,ACTIV_DATE,NUM_MOVES,REF_COUNT,...,MSG_VER,CONTEXT_ID,CF_DATE,CF_EXCHNG,CF_SOURCE,CF_TIME,CF_NAME,DDS_DSO_ID,CF_CURR,SPS_SP_RIC
0,0#COAL-FST-EEX,8205,245,EEX Pwr Coal,0,<NA>,0,<NA>,<NA>,14,...,<NA>,4500,<NA>,464,,<NA>,EEX Pwr Coal,12303,0,.[SPSIDNEM1B008


In [24]:
df = rd.discovery.search(
    view = rd.discovery.Views.SEARCH_ALL,
    filter = "RIC eq '0#CAP=EMCC'",
    select = "DocumentTitle,RIC,PrimaryChainRIC,ExpiryDate",
    top = 1000)
df

,DocumentTitle,RIC
0,European Market Coupling Company (EMCC) Capaci...,0#CAP=EMCC


In [22]:
df['RIC'].iloc[11]

'0#TEST-FINTCEXP-RTE'

In [25]:
rd.get_history('0#CAP=EMCC', count=10)

RDError: Error code -1 | No data to return, please check errors: ERROR: No successful response.
(TS.Interday.UserRequestError.70005, The universe is not found)